In [1]:
import torch
import torchaudio
import librosa
import numpy as np
import parselmouth
from transformers import Wav2Vec2Model, Wav2Vec2Processor

# Load pretrained wav2vec2
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
wav2vec_model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h")
wav2vec_model.eval()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
wav2vec_model.to(device)

SR = 16000

def extract_features(file_path):
    # --- Load audio ---
    y, sr = librosa.load(file_path, sr=SR, mono=True)
    y = y / max(abs(y))
    
    # --- Behavioral features ---
    snd = parselmouth.Sound(file_path)
    pitch = snd.to_pitch()
    pitch_values = pitch.selected_array['frequency']
    pitch_values = pitch_values[pitch_values > 0]
    pitch_mean = np.mean(pitch_values) if len(pitch_values) > 0 else 0
    pitch_std = np.std(pitch_values) if len(pitch_values) > 0 else 0
    
    point_process = parselmouth.praat.call(snd, "To PointProcess (periodic, cc)", 75, 500)
    jitter_local = parselmouth.praat.call(point_process, "Get jitter (local)", 0, 0, 0.0001, 0.02, 1.3)
    shimmer_local = parselmouth.praat.call([snd, point_process], "Get shimmer (local)", 0, 0, 0.0001, 0.02, 1.3, 1.6)
    
    energy = np.mean(y**2)
    
    onset_env = librosa.onset.onset_strength(y=y, sr=sr)
    onsets = librosa.onset.onset_detect(onset_envelope=onset_env, sr=sr)
    duration_sec = librosa.get_duration(y=y, sr=sr)
    speaking_rate = len(onsets)/duration_sec if duration_sec > 0 else 0
    
    behavioral_feats = np.array([pitch_mean, pitch_std, jitter_local, shimmer_local, energy, speaking_rate])
    
    # --- Wav2Vec2 embeddings ---
    input_values = processor(y, sampling_rate=SR, return_tensors="pt", padding=True).input_values.to(device)
    with torch.no_grad():
        embeddings = wav2vec_model(input_values).last_hidden_state
        embeddings = embeddings.mean(dim=1).cpu().numpy().squeeze()  # mean pooling
    
    # --- Combine features ---
    features = np.hstack([embeddings, behavioral_feats])
    return features


C:\Users\admin\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]C:\Users\admin\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\admin\.cache\huggingface\hub\models--facebook--wav2vec2-base-960h. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate D

In [2]:
import os

root_folder = r'User_Voice'

X = []
y_user = []
y_question = []

for user_folder in sorted(os.listdir(root_folder)):
    user_path = os.path.join(root_folder, user_folder)
    if not os.path.isdir(user_path):
        continue
    for file in sorted(os.listdir(user_path)):
        if file.endswith('.wav'):
            file_path = os.path.join(user_path, file)
            features = extract_features(file_path)
            X.append(features)
            
            y_user.append(user_folder)
            q_num = int(file.split('.')[0])  # 01.wav -> 1
            y_question.append(q_num)

X = np.array(X)
y_user = np.array(y_user)
y_question = np.array(y_question)
print("Features ready, shape:", X.shape)


Features ready, shape: (230, 774)


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Split dataset
X_train, X_test, y_train_user, y_test_user, y_train_q, y_test_q = train_test_split(
    X, y_user, y_question, test_size=0.2, stratify=y_user, random_state=42
)

# --- Speaker classifier ---
clf_user = RandomForestClassifier(n_estimators=300, random_state=42)
clf_user.fit(X_train, y_train_user)
y_pred_user = clf_user.predict(X_test)
print("Speaker accuracy:", accuracy_score(y_test_user, y_pred_user))

# --- Question classifier ---
clf_q = RandomForestClassifier(n_estimators=300, random_state=42)
clf_q.fit(X_train, y_train_q)
y_pred_q = clf_q.predict(X_test)
print("Question accuracy:", accuracy_score(y_test_q, y_pred_q))


Speaker accuracy: 0.2608695652173913
Question accuracy: 0.6521739130434783
